# SAM 3 aggressiveness sweep on a clean grain image

This notebook sweeps SAM 3's automatic-mask-generation "aggressiveness" knobs
(`points_per_crop`, `pred_iou_thresh`, `stability_score_thresh`), overlays the resulting
segmentations side by side, and **scores every run against ground truth**.

The clean test images ship with a semantic `_mask.tif` (1 = grain, 2 = grain boundary). Because
grains are separated by boundary pixels, we recover ground-truth grain **instances** with
connected components, which lets us report a proper instance-level metric:

- **Instance precision / recall / F1 @ IoU 0.5** — did each predicted mask land on a distinct real
  grain? Recall rises with aggressiveness; precision falls when SAM invents or merges grains; F1
  is the balance. This is the primary success metric.
- **Foreground IoU / Dice** — pixel-level overlap of *all* predicted grain area vs. the true grain
  region. A prompt-free sanity check that does not depend on instance matching.
- **Grain count** vs. the ~ground-truth count.

Run with the `segmenteverygrain_new_SAM` kernel. `facebook/sam3` is gated — accept the license and
`huggingface-cli login` before the first run.

In [ ]:
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from scipy import ndimage
from tqdm.auto import tqdm
from transformers import pipeline

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

generator = pipeline("mask-generation", model="facebook/sam3", device=device)
print("SAM 3 automatic mask generator ready.")

## 1. Load the image and its ground-truth grains

`GRAIN_CLASS = 1` and `BOUNDARY_CLASS = 2` follow the project's mask convention. GT instances are
the connected components of the grain class (boundaries split them), filtered by `MIN_GRAIN_PX` to
drop specks.

In [ ]:
IMAGE_PATH = "test_only_clean_images/cropped_prac7_etched_020.tif"
MASK_PATH  = "test_only_clean_images/cropped_prac7_etched_020_mask.tif"
GRAIN_CLASS = 1
MIN_GRAIN_PX = 30   # ignore GT/predicted blobs smaller than this

image = Image.open(IMAGE_PATH).convert("RGB")
image_np = np.array(image)
gt = np.array(Image.open(MASK_PATH))


def label_instances(binary, min_px):
    """Connected-component instance labels, small components removed and relabeled 1..N."""
    lbl, n = ndimage.label(binary)
    if n == 0:
        return lbl, 0
    sizes = np.bincount(lbl.ravel())
    keep = np.where(sizes >= min_px)[0]
    keep = keep[keep != 0]
    remap = np.zeros(n + 1, dtype=int)
    remap[keep] = np.arange(1, len(keep) + 1)
    return remap[lbl], len(keep)


gt_label, n_gt = label_instances(gt == GRAIN_CLASS, MIN_GRAIN_PX)
gt_areas = np.bincount(gt_label.ravel(), minlength=n_gt + 1)
print(f"Ground-truth grains: {n_gt}")

fig, ax = plt.subplots(1, 2, figsize=(18, 7))
ax[0].imshow(image_np); ax[0].set_title("Input"); ax[0].axis("off")
ax[1].imshow(np.where(gt_label > 0, gt_label % 20 + 1, 0), cmap="tab20", interpolation="nearest")
ax[1].set_title(f"Ground-truth grain instances (n={n_gt})"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 2. Metrics

`instance_scores` greedily matches each predicted mask to the ground-truth grain it overlaps most,
counting a true positive when IoU ≥ `iou_thr` and each GT grain is claimed at most once.
`foreground_scores` compares the union of all predicted masks against the true grain region.

In [ ]:
def instance_scores(pred_masks, gt_label, gt_areas, n_gt, iou_thr=0.5, min_px=30):
    """Instance precision/recall/F1 by greedy IoU matching to GT connected components."""
    masks = [m for m in pred_masks if int(m.sum()) >= min_px]
    matched, ious, tp = set(), [], 0
    # Match larger masks first so confident, well-formed grains claim their GT partner.
    for m in sorted(masks, key=lambda x: int(x.sum()), reverse=True):
        lbls = gt_label[m]
        lbls = lbls[lbls > 0]
        if lbls.size == 0:
            continue
        cand, inter = np.unique(lbls, return_counts=True)
        pred_area = int(m.sum())
        iou = inter / (pred_area + gt_areas[cand] - inter)
        k = int(np.argmax(iou))
        if iou[k] >= iou_thr and cand[k] not in matched:
            matched.add(int(cand[k])); tp += 1; ious.append(float(iou[k]))
    n_pred = len(masks)
    fp, fn = n_pred - tp, n_gt - len(matched)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(n_pred=n_pred, tp=tp, fp=fp, fn=fn,
               inst_precision=prec, inst_recall=rec, inst_f1=f1,
               mean_matched_iou=float(np.mean(ious)) if ious else 0.0)


def foreground_scores(pred_masks, gt, grain_class=1):
    """Pixel-level overlap of predicted grain area vs. the true grain region."""
    true_fg = gt == grain_class
    pred_fg = np.zeros(true_fg.shape, dtype=bool)
    for m in pred_masks:
        pred_fg |= m
    inter = int((pred_fg & true_fg).sum())
    union = int((pred_fg | true_fg).sum())
    return dict(
        fg_iou=inter / union if union else 0.0,
        fg_dice=2 * inter / (int(pred_fg.sum()) + int(true_fg.sum())) if (pred_fg.sum() + true_fg.sum()) else 0.0,
        fg_precision=inter / int(pred_fg.sum()) if pred_fg.any() else 0.0,
        fg_recall=inter / int(true_fg.sum()) if true_fg.any() else 0.0,
    )


def render_overlay(image_np, masks, alpha=0.5, seed=0):
    """Blend masks as random translucent colors onto the image; return a uint8 RGB array."""
    base = image_np.astype(float) / 255.0
    if base.ndim == 2:
        base = np.stack([base] * 3, axis=-1)
    out = base.copy()
    rng = np.random.default_rng(seed)
    for m in sorted(masks, key=lambda x: int(x.sum()), reverse=True):
        out[m] = (1 - alpha) * out[m] + alpha * rng.random(3)
    return (out * 255).astype(np.uint8)

## 3. Define the parameter sweep

`SWEEP_AXES` holds the two knobs to vary (2 axes → clean grid + heatmaps); `FIXED` are held
constant. Add values to make the sweep finer, or add a third axis (the overlay grid still works;
heatmaps expect exactly two axes). Total runs = product of the axis lengths.

In [ ]:
SWEEP_AXES = {
    "points_per_crop": [32, 64, 96],     # grid density — the main recall lever
    "pred_iou_thresh": [0.88, 0.72, 0.60],  # quality gate — lower keeps more masks
}
FIXED = {
    "points_per_batch": 64,
    "stability_score_thresh": 0.90,
    "stability_score_offset": 1.0,
}

axis_names = list(SWEEP_AXES)
combos = [dict(zip(axis_names, vals)) for vals in itertools.product(*SWEEP_AXES.values())]
print(f"{len(combos)} parameter combinations to run:")
for c in combos:
    print("  ", c)

## 4. Run the sweep

One SAM 3 automatic pass per combination. The overlay image is rendered and stored immediately so
we do not keep every mask in memory. This is the slow cell — runtime ≈ (number of combos) ×
(seconds per pass); raising `points_per_crop` makes each pass slower.

In [ ]:
records, overlays = [], {}
for combo in tqdm(combos, desc="sweep"):
    out = generator(image, **FIXED, **combo)
    masks = [np.asarray(m, dtype=bool) for m in out["masks"]]
    row = dict(combo)
    row.update(instance_scores(masks, gt_label, gt_areas, n_gt, min_px=MIN_GRAIN_PX))
    row.update(foreground_scores(masks, gt, GRAIN_CLASS))
    records.append(row)
    overlays[tuple(combo[a] for a in axis_names)] = render_overlay(image_np, masks)

results = pd.DataFrame(records)
results.to_csv("sam3_sweep_results.csv", index=False)
print("Saved sam3_sweep_results.csv")

In [ ]:
cols = axis_names + ["n_pred", "inst_precision", "inst_recall", "inst_f1",
                     "mean_matched_iou", "fg_iou", "fg_recall"]
ranked = results[cols].sort_values("inst_f1", ascending=False).reset_index(drop=True)
print(f"Ground-truth grains: {n_gt}\n")
print(ranked.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

best = ranked.iloc[0]
print("\nBest by instance F1:")
print("  " + ", ".join(f"{a}={best[a]}" for a in axis_names)
      + f"  ->  F1={best.inst_f1:.3f}, recall={best.inst_recall:.3f},"
      + f" precision={best.inst_precision:.3f}, n_pred={int(best.n_pred)} (GT {n_gt})")

## 5. Overlay every run

Each panel is one parameter combination; the title reports instance F1, recall, and predicted
grain count. Watch coverage grow (and eventually fragment) as the settings get more aggressive.

In [ ]:
res_idx = results.set_index(axis_names)
if len(axis_names) == 2:
    row_vals, col_vals = SWEEP_AXES[axis_names[0]], SWEEP_AXES[axis_names[1]]
else:
    row_vals, col_vals = list(overlays), [None]  # fallback: one column

nr, nc = len(row_vals), len(col_vals)
fig, axes = plt.subplots(nr, nc, figsize=(5.2 * nc, 5.2 * nr), squeeze=False)
for i, rv in enumerate(row_vals):
    for j, cv in enumerate(col_vals):
        key = (rv, cv) if len(axis_names) == 2 else rv
        ax = axes[i][j]
        ax.imshow(overlays[key]); ax.axis("off")
        r = res_idx.loc[key]
        title = (f"{axis_names[0]}={rv}" + (f", {axis_names[1]}={cv}" if len(axis_names) == 2 else "")
                 + f"\nF1={r.inst_f1:.2f}  recall={r.inst_recall:.2f}  n={int(r.n_pred)}")
        ax.set_title(title, fontsize=10)
fig.suptitle(f"SAM 3 aggressiveness sweep (GT = {n_gt} grains)", fontsize=14)
plt.tight_layout(); plt.savefig("sam3_sweep_overlays.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. Metric heatmaps

How each metric varies across the two swept axes. Instance F1 is the one to optimize; recall and
`n_pred` show the aggressiveness climbing, while instance precision shows the cost of over-firing.

In [ ]:
assert len(axis_names) == 2, "Heatmaps need exactly two sweep axes."
metrics_to_map = ["inst_f1", "inst_recall", "inst_precision", "mean_matched_iou", "fg_iou", "n_pred"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, metric in zip(axes.ravel(), metrics_to_map):
    grid = results.pivot(index=axis_names[0], columns=axis_names[1], values=metric)
    im = ax.imshow(grid.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns)
    ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
    ax.set_xlabel(axis_names[1]); ax.set_ylabel(axis_names[0]); ax.set_title(metric)
    for (yy, xx), v in np.ndenumerate(grid.values):
        ax.text(xx, yy, f"{v:.2f}" if metric != "n_pred" else f"{int(v)}",
                ha="center", va="center", color="w", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig("sam3_sweep_heatmaps.png", dpi=130, bbox_inches="tight"); plt.show()

## 7. Metric vs. each parameter

Marginal effect of one knob at a time (averaged over the other), to see which lever moves instance
F1, recall, and precision the most.

In [ ]:
fig, axes = plt.subplots(1, len(axis_names), figsize=(7 * len(axis_names), 5), squeeze=False)
for ax, axis in zip(axes[0], axis_names):
    g = results.groupby(axis)[["inst_f1", "inst_recall", "inst_precision", "fg_iou"]].mean()
    for col in g.columns:
        ax.plot(g.index, g[col], marker="o", label=col)
    ax.set_xlabel(axis); ax.set_ylabel("score (mean over other axis)")
    ax.set_title(f"Effect of {axis}"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 8. Extending this

- **Finer / more axes** — add values to `SWEEP_AXES` or a third knob (e.g. `stability_score_thresh`);
  the overlay grid handles it, heatmaps assume two axes.
- **Beat the internal downscaling** — swap the `generator(...)` call in the sweep for the
  `segment_tiled(...)` helper from `SAM3_clean_grains.ipynb` to segment at native tile resolution;
  the metrics and plots here are unchanged.
- **More images** — wrap sections 1–4 in a loop over the `real_clean_images/*_mask.tif` pairs and
  average the metrics for a less image-specific ranking.